<a href="https://colab.research.google.com/github/rohith154-vsr/ML-2-LAB/blob/main/LP_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
import subprocess
import importlib.util

# check and install wittgenstein library for RIPPER if needed
if importlib.util.find_spec("wittgenstein") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "wittgenstein"])

import pandas as pd
import numpy as np
import math
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import KBinsDiscretizer
import wittgenstein as lw

# load Breast Cancer dataset: target is malignant (0) vs benign (1)
dataset = load_breast_cancer()
df = pd.DataFrame(dataset.data, columns=dataset.feature_names)
df["target"] = dataset.target
df["target"] = df["target"].map({0: "malignant", 1: "benign"})

print("Dataset shape:", df.shape)
print("Class counts:
", df["target"].value_counts())

# RIPPER requires categorical features: discretize continuous variables into 3 quantile bins
X = df.drop(columns=["target"]).copy()
y = df["target"].copy()

discretizer = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="quantile")
X_binned = discretizer.fit_transform(X)
X_binned = pd.DataFrame(X_binned.astype(int).astype(str), columns=X.columns)
X_binned.columns = [col.replace(" ", "_").replace("-", "_") for col in X_binned.columns]

data = X_binned.copy()
data["target"] = y.values

# split 75% train, 25% test stratified by class
train_df, test_df = train_test_split(data, test_size=0.25, random_state=42, stratify=data["target"])

# train RIPPER rule classifier
ripper = lw.RIPPER(random_state=42)
ripper.fit(train_df, class_feat="target", pos_class="malignant")

print("
RIPPER Learned Rules:")
print(ripper.ruleset_)

# evaluate RIPPER on test set
X_test = test_df.drop(columns=["target"])
y_test = test_df["target"]

predictions = ripper.predict(X_test)
predictions = np.asarray(predictions).reshape(-1)
predictions = ["malignant" if p == 1 else "benign" for p in predictions]

print(f"
RIPPER Test Accuracy: {round(accuracy_score(y_test, predictions) * 100, 2)}%")
print(classification_report(y_test, predictions, zero_division=0))

# -------------------------------------------------------------------------
# FOIL-style rule learning: implemented from scratch using FOIL Information Gain
# Formula: Gain(L) = p1 * (log2(p1 / (p1 + n1)) - log2(p0 / (p0 + n0)))
# where p0, n0 = positive/negative coverage before adding literal
#       p1, n1 = positive/negative coverage after adding candidate literal
# -------------------------------------------------------------------------
def foil_info_gain(p0, n0, p1, n1):
    if p1 == 0:
        return -float("inf")
    gain = p1 * (math.log2(p1 / (p1 + n1) + 1e-9) - math.log2(p0 / (p0 + n0) + 1e-9))
    return gain

def learn_foil_rules(data, target_col, pos_class, feature_cols, max_rules=3, max_conditions=3):
    remaining = data.copy()
    rules = []

    # sequential covering loop: learn rules until positives are covered
    for _ in range(max_rules):
        pos_total = (remaining[target_col] == pos_class).sum()
        if pos_total == 0:
            break

        rule_conditions = []
        current_data = remaining.copy()

        # greedily add conditions that maximize FOIL info gain
        for _ in range(max_conditions):
            p0 = (current_data[target_col] == pos_class).sum()
            n0 = (current_data[target_col] != pos_class).sum()
            if n0 == 0:
                break # pure rule achieved

            best_gain = -float("inf")
            best_cond = None
            best_mask = None

            for feat in feature_cols:
                for val in current_data[feat].unique():
                    mask = current_data[feat] == val
                    subset = current_data[mask]
                    p1 = (subset[target_col] == pos_class).sum()
                    n1 = (subset[target_col] != pos_class).sum()
                    gain = foil_info_gain(p0, n0, p1, n1)
                    if gain > best_gain:
                        best_gain = gain
                        best_cond = (feat, val)
                        best_mask = mask

            if best_cond is None or best_gain <= 0:
                break

            rule_conditions.append(best_cond)
            current_data = current_data[best_mask]

        if not rule_conditions:
            break

        # eliminate positive instances covered by this rule
        mask = pd.Series([True] * len(remaining), index=remaining.index)
        for feat, val in rule_conditions:
            mask &= remaining[feat] == val
        covered_pos = mask & (remaining[target_col] == pos_class)
        remaining = remaining[~covered_pos]

        rules.append(rule_conditions)

    return rules

feature_cols = [c for c in train_df.columns if c != "target"]
foil_rules = learn_foil_rules(train_df, "target", "malignant", feature_cols)

print("
FOIL-style Learned Rules:")
for i, rule in enumerate(foil_rules, 1):
    conditions = " AND ".join([f"{f}={v}" for f, v in rule])
    print(f"  Rule {i}: IF {conditions} THEN target=malignant")

# evaluate FOIL rules on test set
def apply_foil_rules(rules, X_test, pos_class="malignant", neg_class="benign"):
    preds = []
    for _, row in X_test.iterrows():
        classified = False
        for rule in rules:
            if all(row[f] == v for f, v in rule):
                preds.append(pos_class)
                classified = True
                break
        if not classified:
            preds.append(neg_class)
    return preds

foil_preds = apply_foil_rules(foil_rules, X_test)
print(f"
FOIL Test Accuracy: {round(accuracy_score(y_test, foil_preds) * 100, 2)}%")
print(classification_report(y_test, foil_preds, zero_division=0))